# CBRAIN OASIS Preprocessing Freeze (Step 2B)

This notebook freezes leakage-safe preprocessing for baseline v1 using train-only fit statistics.

In [ ]:
# 1) Load Step 2A artifacts
from __future__ import annotations

import csv
import json
from datetime import date
from pathlib import Path

cwd = Path.cwd().resolve()
candidate_roots = [cwd]
if cwd.name == 'notebooks':
    candidate_roots.append(cwd.parent)
else:
    candidate_roots.append(cwd / 'projects' / 'cbrain_oasis_cognition')
    candidate_roots.append(cwd.parent)

repo_root = None
for candidate in candidate_roots:
    if (candidate / 'results' / 'cohort' / 'baseline_cohort_full.csv').exists():
        repo_root = candidate
        break

if repo_root is None:
    raise FileNotFoundError('Could not locate Step 2A artifacts under results/cohort/.')

cohort_dir = repo_root / 'results' / 'cohort'
pre_dir = repo_root / 'results' / 'preprocessing'
docs_path = repo_root / 'docs' / 'preprocessing_freeze.md'

baseline_cohort_full_path = cohort_dir / 'baseline_cohort_full.csv'
split_subjects_path = cohort_dir / 'split_subjects.json'

with baseline_cohort_full_path.open('r', encoding='utf-8', newline='') as f:
    reader = csv.DictReader(f)
    baseline_rows = [dict(r) for r in reader]

split_payload = json.loads(split_subjects_path.read_text(encoding='utf-8'))

print(f'Repo root: {repo_root}')
print(f'Baseline rows loaded: {len(baseline_rows)}')
print(f'Train subjects in split payload: {split_payload.get('train_subject_count')}')
print(f'Validation subjects in split payload: {split_payload.get('validation_subject_count')}')

In [ ]:
# 2) Validate split contracts and build train/validation cohorts
required_split_keys = [
    'train_subject_ids',
    'validation_subject_ids',
    'train_subject_count',
    'validation_subject_count',
    'overlap_count',
]
missing_split_keys = [k for k in required_split_keys if k not in split_payload]
if missing_split_keys:
    raise AssertionError(f'split_subjects.json missing keys: {missing_split_keys}')

if split_payload['overlap_count'] != 0:
    raise AssertionError('Leakage gate failed: split_subjects overlap_count must be 0.')

train_subject_ids = sorted(split_payload['train_subject_ids'])
validation_subject_ids = sorted(split_payload['validation_subject_ids'])

if len(train_subject_ids) != split_payload['train_subject_count']:
    raise AssertionError('train_subject_count mismatch vs subject-id list length.')
if len(validation_subject_ids) != split_payload['validation_subject_count']:
    raise AssertionError('validation_subject_count mismatch vs subject-id list length.')

overlap = sorted(set(train_subject_ids).intersection(validation_subject_ids))
if overlap:
    raise AssertionError(f'Leakage gate failed: subject overlap found: {overlap}')

baseline_by_subject = {}
for row in baseline_rows:
    sid = row['Subject.ID']
    if sid in baseline_by_subject:
        raise AssertionError(f'Duplicate Subject.ID in baseline cohort: {sid}')
    baseline_by_subject[sid] = row

missing_train_subjects = [sid for sid in train_subject_ids if sid not in baseline_by_subject]
missing_val_subjects = [sid for sid in validation_subject_ids if sid not in baseline_by_subject]
if missing_train_subjects or missing_val_subjects:
    raise AssertionError('Split subjects missing from baseline cohort.')

train_rows = [baseline_by_subject[sid] for sid in train_subject_ids]
validation_rows = [baseline_by_subject[sid] for sid in validation_subject_ids]

print(f'Train rows: {len(train_rows)}')
print(f'Validation rows: {len(validation_rows)}')

In [ ]:
# 3) Define schema, exclusions, and strict leakage policy
label_col = 'cognitive_impairment'
numeric_predictors = ['Age', 'EDUC', 'SES', 'eTIV', 'nWBV', 'ASF']
categorical_predictors = ['Gender']
candidate_predictors = ['Age', 'Gender', 'EDUC', 'SES', 'eTIV', 'nWBV', 'ASF']
locked_exclusions = ['CDR', 'Group', 'MMSE', 'Subject.ID', 'MRI.ID', 'rownames', 'Hand', 'Visit', 'MR.Delay']

required_columns = set(candidate_predictors + [label_col, 'Subject.ID'])
available_columns = set(baseline_rows[0].keys())
missing_cols = sorted(required_columns - available_columns)
if missing_cols:
    raise AssertionError(f'Missing required columns in baseline cohort: {missing_cols}')

for col in locked_exclusions:
    if col in candidate_predictors:
        raise AssertionError(f'Predictor list includes excluded leakage column: {col}')

print('Predictors:', candidate_predictors)
print('Label:', label_col)

In [ ]:
# 4) Fit preprocessing on train only (imputation, encoding, scaling)
def _is_missing(v: str) -> bool:
    return str(v).strip() == ''

def _to_float(v: str) -> float:
    return float(v)

def _median(values: list[float]) -> float:
    ordered = sorted(values)
    n = len(ordered)
    if n == 0:
        raise AssertionError('Cannot compute median of empty list.')
    mid = n // 2
    if n % 2 == 1:
        return ordered[mid]
    return (ordered[mid - 1] + ordered[mid]) / 2.0

ses_train_non_missing = [
    _to_float(r['SES']) for r in train_rows if not _is_missing(r['SES'])
]
ses_median = _median(ses_train_non_missing)

gender_categories_train = sorted({str(r['Gender']) for r in train_rows})
if not gender_categories_train:
    raise AssertionError('No gender categories available in train.')
gender_to_code = {cat: idx for idx, cat in enumerate(gender_categories_train)}

numeric_after_impute = ['Age', 'EDUC', 'SES_imputed', 'eTIV', 'nWBV', 'ASF']
means = {}
stds = {}

def _train_numeric_value(row: dict, name: str) -> float:
    if name == 'SES_imputed':
        return ses_median if _is_missing(row['SES']) else _to_float(row['SES'])
    raw = row[name]
    if _is_missing(raw):
        raise AssertionError(f'Missing value in train for required predictor {name}.')
    return _to_float(raw)

for feat in numeric_after_impute:
    vals = [_train_numeric_value(r, feat) for r in train_rows]
    mean_val = sum(vals) / len(vals)
    var = sum((x - mean_val) ** 2 for x in vals) / len(vals)
    std_val = var ** 0.5
    if std_val == 0.0:
        std_val = 1.0
    means[feat] = mean_val
    stds[feat] = std_val

fit_payload = {
    'label_column': label_col,
    'candidate_predictors': candidate_predictors,
    'numeric_predictors_raw': numeric_predictors,
    'categorical_predictors_raw': categorical_predictors,
    'imputation': {'SES': {'strategy': 'median', 'value': ses_median}},
    'categorical_encoding': {'Gender': {'mapping': gender_to_code, 'train_categories': gender_categories_train}},
    'scaling': {feat: {'mean': means[feat], 'std': stds[feat]} for feat in numeric_after_impute},
}

print('SES train median:', ses_median)
print('Gender mapping:', gender_to_code)

In [ ]:
# 5) Apply frozen preprocessing to train and validation
final_feature_order = [
    'Age_z',
    'Gender_code',
    'EDUC_z',
    'SES_imputed_z',
    'SES_missing',
    'eTIV_z',
    'nWBV_z',
    'ASF_z',
]

def _transform_row(row: dict, split_name: str) -> tuple[dict, int]:
    sid = row['Subject.ID']

    ses_missing = 1 if _is_missing(row['SES']) else 0
    ses_imputed = ses_median if ses_missing else _to_float(row['SES'])

    gender_raw = str(row['Gender'])
    if gender_raw not in gender_to_code:
        raise AssertionError(f'Unknown Gender category in {split_name} for {sid}: {gender_raw}')
    gender_code = gender_to_code[gender_raw]

    raw_numeric = {
        'Age': _to_float(row['Age']),
        'EDUC': _to_float(row['EDUC']),
        'SES_imputed': ses_imputed,
        'eTIV': _to_float(row['eTIV']),
        'nWBV': _to_float(row['nWBV']),
        'ASF': _to_float(row['ASF']),
    }

    features = {
        'Age_z': (raw_numeric['Age'] - means['Age']) / stds['Age'],
        'Gender_code': int(gender_code),
        'EDUC_z': (raw_numeric['EDUC'] - means['EDUC']) / stds['EDUC'],
        'SES_imputed_z': (raw_numeric['SES_imputed'] - means['SES_imputed']) / stds['SES_imputed'],
        'SES_missing': int(ses_missing),
        'eTIV_z': (raw_numeric['eTIV'] - means['eTIV']) / stds['eTIV'],
        'nWBV_z': (raw_numeric['nWBV'] - means['nWBV']) / stds['nWBV'],
        'ASF_z': (raw_numeric['ASF'] - means['ASF']) / stds['ASF'],
    }

    label_raw = row[label_col]
    if _is_missing(label_raw):
        raise AssertionError(f'Missing label for {sid}')
    label = int(label_raw)
    if label not in (0, 1):
        raise AssertionError(f'Invalid label for {sid}: {label_raw}')

    return features, label

def _transform_dataset(rows_subset: list[dict], split_name: str):
    X_rows = []
    y_rows = []
    for row in rows_subset:
        feats, lab = _transform_row(row, split_name=split_name)
        X_rows.append(feats)
        y_rows.append({'cognitive_impairment': lab})
    return X_rows, y_rows

X_train, y_train = _transform_dataset(train_rows, split_name='train')
X_validation, y_validation = _transform_dataset(validation_rows, split_name='validation')

def _count_missing_output(X_rows: list[dict]) -> dict:
    counts = {k: 0 for k in final_feature_order}
    for row in X_rows:
        for k in final_feature_order:
            v = row[k]
            if v is None or (isinstance(v, str) and v.strip() == ''):
                counts[k] += 1
    return counts

missing_train_out = _count_missing_output(X_train)
missing_val_out = _count_missing_output(X_validation)

if any(v != 0 for v in missing_train_out.values()):
    raise AssertionError(f'Missing output values in train matrix: {missing_train_out}')
if any(v != 0 for v in missing_val_out.values()):
    raise AssertionError(f'Missing output values in validation matrix: {missing_val_out}')

print('Transformed train rows:', len(X_train))
print('Transformed validation rows:', len(X_validation))

In [ ]:
# 6) Materialize frozen preprocessing artifacts
pre_dir.mkdir(parents=True, exist_ok=True)

x_train_path = pre_dir / 'X_train.csv'
x_validation_path = pre_dir / 'X_validation.csv'
y_train_path = pre_dir / 'y_train.csv'
y_validation_path = pre_dir / 'y_validation.csv'
feature_manifest_path = pre_dir / 'feature_manifest.json'
preprocessing_fit_path = pre_dir / 'preprocessing_fit.json'
preprocessing_summary_path = pre_dir / 'preprocessing_summary.json'

with x_train_path.open('w', encoding='utf-8', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=final_feature_order)
    writer.writeheader()
    for row in X_train:
        writer.writerow({k: row[k] for k in final_feature_order})

with x_validation_path.open('w', encoding='utf-8', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=final_feature_order)
    writer.writeheader()
    for row in X_validation:
        writer.writerow({k: row[k] for k in final_feature_order})

with y_train_path.open('w', encoding='utf-8', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=['cognitive_impairment'])
    writer.writeheader()
    writer.writerows(y_train)

with y_validation_path.open('w', encoding='utf-8', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=['cognitive_impairment'])
    writer.writeheader()
    writer.writerows(y_validation)

feature_manifest = {
    'label_column': label_col,
    'raw_candidate_predictors': candidate_predictors,
    'raw_numeric_predictors': numeric_predictors,
    'raw_categorical_predictors': categorical_predictors,
    'locked_exclusions': locked_exclusions,
    'final_feature_order': final_feature_order,
    'fit_on_train_only': True,
    'row_alignment_rule': 'Rows follow sorted subject ids from split_subjects.json; y files align row-wise with X files.',
}
feature_manifest_path.write_text(json.dumps(feature_manifest, indent=2, sort_keys=True) + '\n', encoding='utf-8')

preprocessing_fit_path.write_text(json.dumps(fit_payload, indent=2, sort_keys=True) + '\n', encoding='utf-8')

def _label_stats(y_rows: list[dict]) -> dict:
    positives = sum(int(r['cognitive_impairment']) for r in y_rows)
    total = len(y_rows)
    negatives = total - positives
    return {
        'row_count': total,
        'positive_count': positives,
        'negative_count': negatives,
        'positive_rate': (positives / total) if total else 0.0,
    }

train_stats = _label_stats(y_train)
val_stats = _label_stats(y_validation)

summary_payload = {
    'train_row_count': len(X_train),
    'validation_row_count': len(X_validation),
    'train_subject_count': len(train_subject_ids),
    'validation_subject_count': len(validation_subject_ids),
    'overlap_count': len(overlap),
    'train_label_distribution': train_stats,
    'validation_label_distribution': val_stats,
    'missing_counts_transformed_X_train': missing_train_out,
    'missing_counts_transformed_X_validation': missing_val_out,
    'final_feature_order': final_feature_order,
}
preprocessing_summary_path.write_text(json.dumps(summary_payload, indent=2, sort_keys=True) + '\n', encoding='utf-8')

run_date = date.today().isoformat()
docs_text = f"""# Preprocessing Freeze (Step 2B)

Date: {run_date}  
Notebook: `notebooks/03_preprocessing_freeze.ipynb`  
Inputs: `results/cohort/baseline_cohort_full.csv`, `results/cohort/split_subjects.json`

## Leakage-Safe Fit Protocol

All preprocessing parameters are fit on **train subjects only** and then applied unchanged to validation.

- Split rule source: `results/cohort/split_subjects.json`
- Subject overlap check: **{len(overlap)}**
- Leakage fields in model matrix: **not present**

## Schema and Transform Rules

Raw candidate predictors:
- `Age`, `Gender`, `EDUC`, `SES`, `eTIV`, `nWBV`, `ASF`

Locked exclusions:
- `CDR`, `Group`, `MMSE`, `Subject.ID`, `MRI.ID`, `rownames`, `Hand`, `Visit`, `MR.Delay`

Applied transforms:
- `SES`: median imputation from train only (median={ses_median})
- `SES_missing`: binary missingness indicator
- `Gender`: ordinal encoding using train categories {gender_categories_train} with mapping {gender_to_code}
- numeric standardization (z-score) fit on train only for `Age`, `EDUC`, `SES_imputed`, `eTIV`, `nWBV`, `ASF`

Final feature order:
- {', '.join(f'`{f}`' for f in final_feature_order)}

## Train/Validation Label Balance

| Split | Rows | Positive | Negative | Positive Rate |
|---|---:|---:|---:|---:|
| Train | {train_stats['row_count']} | {train_stats['positive_count']} | {train_stats['negative_count']} | {train_stats['positive_rate']:.4f} |
| Validation | {val_stats['row_count']} | {val_stats['positive_count']} | {val_stats['negative_count']} | {val_stats['positive_rate']:.4f} |

## Frozen Artifacts

- `results/preprocessing/X_train.csv`
- `results/preprocessing/X_validation.csv`
- `results/preprocessing/y_train.csv`
- `results/preprocessing/y_validation.csv`
- `results/preprocessing/feature_manifest.json`
- `results/preprocessing/preprocessing_fit.json`
- `results/preprocessing/preprocessing_summary.json`
"""
docs_path.write_text(docs_text, encoding='utf-8')

print('Wrote:', x_train_path)
print('Wrote:', x_validation_path)
print('Wrote:', y_train_path)
print('Wrote:', y_validation_path)
print('Wrote:', feature_manifest_path)
print('Wrote:', preprocessing_fit_path)
print('Wrote:', preprocessing_summary_path)
print('Wrote:', docs_path)